# Task 6 — Customer Segmentation (RFM + K-Means + SHAP)
Kết nối MotherDuck → Tính RFM → K-Means Clustering → SHAP → Lưu kết quả

In [ ]:
# Cell 1: Cài thư viện
!pip install duckdb scikit-learn shap plotly -q

In [ ]:
# Cell 2: Kết nối MotherDuck
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Dán token MotherDuck vào đây
MOTHERDUCK_TOKEN = "paste_token_cua_ban_vao_day"

conn = duckdb.connect(f"md:my_db?motherduck_token={MOTHERDUCK_TOKEN}")

# Lấy dữ liệu từ Gold layer
df = conn.execute("""
    SELECT *
    FROM main_gold.gold_customer_features
    WHERE total_purchases > 0
""").df()

print(f"Số khách hàng đã mua hàng: {len(df):,}")
df.head()

In [ ]:
# Cell 3: Tính RFM Score
# Recency: càng nhỏ càng tốt (mua gần đây) → đảo thứ tự
df['R_score'] = pd.qcut(df['recency_days'], q=5, labels=[5,4,3,2,1], duplicates='drop')

# Frequency và Monetary: càng cao càng tốt
df['F_score'] = pd.qcut(df['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])
df['M_score'] = pd.qcut(df['monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5])

df['R_score'] = df['R_score'].astype(int)
df['F_score'] = df['F_score'].astype(int)
df['M_score'] = df['M_score'].astype(int)
df['RFM_score'] = df['R_score'] * 100 + df['F_score'] * 10 + df['M_score']

def rfm_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4: return 'Champions'
    elif r >= 3 and f >= 3:           return 'Loyal Customers'
    elif r >= 4 and f <= 2:           return 'New Customers'
    elif r <= 2 and f >= 3 and m >= 3:return 'At Risk'
    elif r <= 2 and f <= 2:           return 'Lost'
    else:                             return 'Potential'

df['rfm_segment'] = df.apply(rfm_segment, axis=1)
print(df['rfm_segment'].value_counts())

In [ ]:
# Cell 4: Elbow Method để chọn k tối ưu
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features_cols = [
    'recency_days', 'frequency', 'monetary',
    'view_to_cart_rate', 'cart_to_purchase_rate', 'avg_order_value'
]

X = df[features_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias, silhouettes = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(K_range, inertias, 'bo-')
ax1.set_xlabel('Số cụm K'); ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')
ax2.plot(K_range, silhouettes, 'ro-')
ax2.set_xlabel('Số cụm K'); ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score')
plt.tight_layout()
plt.show()

best_k = K_range[silhouettes.index(max(silhouettes))]
print(f"K tối ưu theo Silhouette: {best_k}")

In [ ]:
# Cell 5: Train K-Means với k=4
K_OPTIMAL = 4

kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Xem profile từng cụm
profile = df.groupby('cluster')[features_cols].mean().round(2)
print(profile)

# Gán nhãn kinh doanh (điều chỉnh sau khi xem profile)
cluster_labels = {
    0: 'VIP — Mua nhiều, chi nhiều',
    1: 'Tiềm năng — Tích cực nhưng ít mua',
    2: 'Thụ động — Ít tương tác',
    3: 'Rủi ro — Lâu không quay lại'
}
df['cluster_label'] = df['cluster'].map(cluster_labels)
print("\nPhân bố cụm:")
print(df['cluster_label'].value_counts())

In [ ]:
# Cell 6: Visualize clustering
import plotly.express as px

fig = px.scatter(
    df.sample(5000, random_state=42),
    x='recency_days',
    y='monetary',
    color='cluster_label',
    size='frequency',
    hover_data=['user_id', 'total_purchases'],
    title='Customer Segments — RFM Scatter Plot',
    labels={'recency_days': 'Recency (days)', 'monetary': 'Monetary (USD)'}
)
fig.show()

In [ ]:
# Cell 7: SHAP Explanation
import shap
from sklearn.ensemble import RandomForestClassifier

# Train RF để SHAP có thể giải thích
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_scaled, df['cluster'])

# SHAP
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_scaled[:1000])

shap.summary_plot(
    shap_values,
    pd.DataFrame(X_scaled[:1000], columns=features_cols),
    plot_type='bar',
    class_names=list(cluster_labels.values())
)

In [ ]:
# Cell 8: Lưu kết quả vào MotherDuck
result_df = df[[
    'user_id', 'cluster', 'cluster_label',
    'R_score', 'F_score', 'M_score',
    'RFM_score', 'rfm_segment'
]].copy()

conn.execute("DROP TABLE IF EXISTS main_gold.cluster_predictions")
conn.register('result_df', result_df)
conn.execute("""
    CREATE TABLE main_gold.cluster_predictions AS
    SELECT * FROM result_df
""")

count = conn.execute("SELECT COUNT(*) FROM main_gold.cluster_predictions").fetchone()[0]
print(f"✅ Đã lưu {count:,} rows vào cluster_predictions")
conn.close()